In [1]:
import pandas as pd
import numpy as np

import glob
import os
import os.path
import re
import sys
import warnings

from scipy.spatial import KDTree
from scipy.spatial import distance

#-------------------------------
from io import StringIO


import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap, ScalarMappable
from matplotlib.patches import Rectangle
import matplotlib.colors as colors

#from mpl_toolkits.mplot3d import Axes3D

In [2]:
#def main():

# ignore warnings
warnings.filterwarnings("ignore")

In [3]:
def layer_category_fix_n(zmin,zmax,n):
    layers_list = []
    beginning = zmin
    h_step = (zmax-zmin)/n
    n_layer = 0
    while (beginning < zmax):
        layers_list.append([beginning, beginning+h_step, h_step, n_layer])
        beginning = beginning + h_step 
        n_layer += 1
    return sorted(layers_list)

In [8]:
def bond_length_calc_mean():
    
    # ignore warnings
    warnings.filterwarnings("ignore")
    
    #layer_height = int(sys.argv[-1])
    layer_height = 2
            
    #Creatte directory to put avg results
    directory_frames = f'cascades_form_spectra_for_movie_npt/results_bonds_data_vs_z_h{layer_height}_shrinking'
    directory = f'{directory_frames}/results_bonds_vs_z_h{layer_height}_shrinking'
    try:
        os.makedirs(directory)
    except FileExistsError:
        # directory already exists
        pass
    

    # list of all files
    list_files = glob.glob(f'{directory_frames}/bonds_lenght_data_h2_layer*_shrinking_final_pyro_cascades_simulation*.xyz')
    #list_files = [list_files[0]]
    
    list_sim = []
    for filename in list_files:
        list_sim.append(re.search('simulation(.+?).xyz', filename).group(1))
        
    list_sim = list(set(list_sim))
    #list_sim = [list_sim[0]]
    print(list_sim)
        
    
    for simulation in list_sim:
        print(simulation)
        # load data from file
        filename = f"cascades_form_spectra_for_movie_npt/final_pyro_cascades_simulation{simulation}.xyz"
        input_data = open(filename).read()
        df_atoms = pd.read_table(filename, skiprows=9,
                                       names=['element', 'x', 'y', 'z', 'id', 'coord_num', 'pot_eng'],
                                       sep='\\s+')
        df_atoms = df_atoms.drop(['element'],axis=1)
            
                       
        # assign z_layer_fix_n
        init_intf = 6
        init_z_min = -169.64307163528616
        init_z_max = 150.98293179686004
        n = (init_z_max - init_z_min) //layer_height
            
        layers = layer_category_fix_n(df_atoms['z'].min(),df_atoms['z'].max(), n)        
        #print(layers)
        array=np.array(layers)        
        df_final = pd.DataFrame(array,columns=['beginning','ending','height','z_layers'])
        df_final['mean_bond_length'] = 0
        df_final['disp_bond_length'] = 0
        
         # list of files for certain simulation
        list_files = glob.glob(f'{directory_frames}/bonds_lenght_data_h2_layer*_shrinking_final_pyro_cascades_simulation{simulation}.xyz')
        count_files = 1
        for filename in list_files:
            n = 73
            print(count_files, "/", len(list_files), "file name:", filename[n:])            
            layer = re.search('layer(.+?)_shrinking', filename).group(1)
        
            #load bond length data
            df_bonds = pd.read_table(filename, header=0, sep=',', index_col=0)
            
            #for i in df_bonds.index:    
            #    dist = distance.cdist(np.array([df_bonds.loc[i,['x_central_atom','y_central_atom','z_central_atom']].to_numpy()]),
            #                          np.array([df_bonds.loc[i,['x_neigb1_atom','y_neigb1_atom','z_neigb1_atom']].to_numpy()]),
            #                          metric='euclidean')
            #    
            #    df_bonds.loc[i,'length'] = dist[0,0]
            
            a = df_bonds[['x_central_atom','y_central_atom','z_central_atom']].to_numpy()
            b = df_bonds[['x_neigb1_atom','y_neigb1_atom','z_neigb1_atom']].to_numpy()

            dist = np.sqrt((a[:,0] - b[:,0])**2 + (a[:,1] - b[:,1])**2 + (a[:,2] - b[:,2])**2)
            df_bonds['length'] = dist
            
            bond_mean = df_bonds['length'].mean()
            bond_median = df_bonds['length'].median()
            bond_min = df_bonds['length'].min()
            bond_max = df_bonds['length'].max()
            bond_first_quantile = df_bonds['length'].quantile(q=0.25)
            bond_third_quantile = df_bonds['length'].quantile(q=0.75)
            bond_disp = df_bonds['length'].std(ddof=1)
            
            df_final.loc[df_final['z_layers'] == float(layer), 'mean_bond_length'] = bond_mean
            df_final.loc[df_final['z_layers'] == float(layer), 'median_bond_length'] = bond_median
            df_final.loc[df_final['z_layers'] == float(layer), 'disp_bond_length'] = bond_disp
            df_final.loc[df_final['z_layers'] == float(layer), 'min_bond_length'] = bond_min
            df_final.loc[df_final['z_layers'] == float(layer), 'max_bond_length'] = bond_max
            df_final.loc[df_final['z_layers'] == float(layer), 'first_quantile_bond_length'] = bond_first_quantile
            df_final.loc[df_final['z_layers'] == float(layer), 'third_quantile_bond_length'] = bond_third_quantile
            #print (layer, bond_mean, bond_disp)
            #display(df_final[df_final['z_layers'] == float(layer)])
        
            count_files += 1
        
        border_factor = 0
        if border_factor == 0:
        #    file_output = f'{directory}/bonds_vs_z_h{layer_height}_shrinking_{filename[n:]}'
            file_length = f'{directory}/bonds_lenght_h{layer_height}_shrinking_final_pyro_cascades_simulation{simulation}.xyz'
        #    file_angl = f'{directory}/bonds_angl_h{layer_height}_shrinking_{filename[n:]}'
        else:
        #    file_output = f'{directory}/bonds_vs_z_h{layer_height}_b{border_factor}_shrinking_{filename[n:]}'
            file_length = f'{directory}/bonds_lenght_h{layer_height}_b{border_factor}_shrinking_final_pyro_cascades_simulation{simulation}.xyz'
        #    file_angl = f'{directory}/bonds_angl_h{layer_height}_b{border_factor}_shrinking_{filename[n:]}'
        
        
        df_final.to_csv(file_length, sep=',')
        print("Done")

In [9]:
bond_length_calc_mean()

['350', '500', '200', '0']
350
1 / 161 file name: bonds_lenght_data_h2_layer148.0_shrinking_final_pyro_cascades_simulation350.xyz
2 / 161 file name: bonds_lenght_data_h2_layer19.0_shrinking_final_pyro_cascades_simulation350.xyz
3 / 161 file name: bonds_lenght_data_h2_layer5.0_shrinking_final_pyro_cascades_simulation350.xyz
4 / 161 file name: bonds_lenght_data_h2_layer74.0_shrinking_final_pyro_cascades_simulation350.xyz
5 / 161 file name: bonds_lenght_data_h2_layer27.0_shrinking_final_pyro_cascades_simulation350.xyz
6 / 161 file name: bonds_lenght_data_h2_layer105.0_shrinking_final_pyro_cascades_simulation350.xyz
7 / 161 file name: bonds_lenght_data_h2_layer52.0_shrinking_final_pyro_cascades_simulation350.xyz
8 / 161 file name: bonds_lenght_data_h2_layer144.0_shrinking_final_pyro_cascades_simulation350.xyz
9 / 161 file name: bonds_lenght_data_h2_layer131.0_shrinking_final_pyro_cascades_simulation350.xyz
10 / 161 file name: bonds_lenght_data_h2_layer135.0_shrinking_final_pyro_cascades_si

19 / 160 file name: bonds_lenght_data_h2_layer60.0_shrinking_final_pyro_cascades_simulation0.xyz
20 / 160 file name: bonds_lenght_data_h2_layer42.0_shrinking_final_pyro_cascades_simulation0.xyz
21 / 160 file name: bonds_lenght_data_h2_layer74.0_shrinking_final_pyro_cascades_simulation0.xyz
22 / 160 file name: bonds_lenght_data_h2_layer45.0_shrinking_final_pyro_cascades_simulation0.xyz
23 / 160 file name: bonds_lenght_data_h2_layer127.0_shrinking_final_pyro_cascades_simulation0.xyz
24 / 160 file name: bonds_lenght_data_h2_layer54.0_shrinking_final_pyro_cascades_simulation0.xyz
25 / 160 file name: bonds_lenght_data_h2_layer3.0_shrinking_final_pyro_cascades_simulation0.xyz
26 / 160 file name: bonds_lenght_data_h2_layer117.0_shrinking_final_pyro_cascades_simulation0.xyz
27 / 160 file name: bonds_lenght_data_h2_layer65.0_shrinking_final_pyro_cascades_simulation0.xyz
28 / 160 file name: bonds_lenght_data_h2_layer57.0_shrinking_final_pyro_cascades_simulation0.xyz
29 / 160 file name: bonds_len

105 / 160 file name: bonds_lenght_data_h2_layer145.0_shrinking_final_pyro_cascades_simulation0.xyz
106 / 160 file name: bonds_lenght_data_h2_layer21.0_shrinking_final_pyro_cascades_simulation0.xyz
107 / 160 file name: bonds_lenght_data_h2_layer131.0_shrinking_final_pyro_cascades_simulation0.xyz
108 / 160 file name: bonds_lenght_data_h2_layer80.0_shrinking_final_pyro_cascades_simulation0.xyz
109 / 160 file name: bonds_lenght_data_h2_layer7.0_shrinking_final_pyro_cascades_simulation0.xyz
110 / 160 file name: bonds_lenght_data_h2_layer90.0_shrinking_final_pyro_cascades_simulation0.xyz
111 / 160 file name: bonds_lenght_data_h2_layer1.0_shrinking_final_pyro_cascades_simulation0.xyz
112 / 160 file name: bonds_lenght_data_h2_layer103.0_shrinking_final_pyro_cascades_simulation0.xyz
113 / 160 file name: bonds_lenght_data_h2_layer146.0_shrinking_final_pyro_cascades_simulation0.xyz
114 / 160 file name: bonds_lenght_data_h2_layer84.0_shrinking_final_pyro_cascades_simulation0.xyz
115 / 160 file nam

29 / 161 file name: bonds_lenght_data_h2_layer62.0_shrinking_final_pyro_cascades_simulation350.xyz
30 / 161 file name: bonds_lenght_data_h2_layer37.0_shrinking_final_pyro_cascades_simulation350.xyz
31 / 161 file name: bonds_lenght_data_h2_layer50.0_shrinking_final_pyro_cascades_simulation350.xyz
32 / 161 file name: bonds_lenght_data_h2_layer16.0_shrinking_final_pyro_cascades_simulation350.xyz
33 / 161 file name: bonds_lenght_data_h2_layer98.0_shrinking_final_pyro_cascades_simulation350.xyz
34 / 161 file name: bonds_lenght_data_h2_layer58.0_shrinking_final_pyro_cascades_simulation350.xyz
35 / 161 file name: bonds_lenght_data_h2_layer3.0_shrinking_final_pyro_cascades_simulation350.xyz
36 / 161 file name: bonds_lenght_data_h2_layer0.0_shrinking_final_pyro_cascades_simulation350.xyz
37 / 161 file name: bonds_lenght_data_h2_layer134.0_shrinking_final_pyro_cascades_simulation350.xyz
38 / 161 file name: bonds_lenght_data_h2_layer13.0_shrinking_final_pyro_cascades_simulation350.xyz
39 / 161 fi

115 / 161 file name: bonds_lenght_data_h2_layer122.0_shrinking_final_pyro_cascades_simulation350.xyz
116 / 161 file name: bonds_lenght_data_h2_layer120.0_shrinking_final_pyro_cascades_simulation350.xyz
117 / 161 file name: bonds_lenght_data_h2_layer92.0_shrinking_final_pyro_cascades_simulation350.xyz
118 / 161 file name: bonds_lenght_data_h2_layer104.0_shrinking_final_pyro_cascades_simulation350.xyz
119 / 161 file name: bonds_lenght_data_h2_layer17.0_shrinking_final_pyro_cascades_simulation350.xyz
120 / 161 file name: bonds_lenght_data_h2_layer107.0_shrinking_final_pyro_cascades_simulation350.xyz
121 / 161 file name: bonds_lenght_data_h2_layer60.0_shrinking_final_pyro_cascades_simulation350.xyz
122 / 161 file name: bonds_lenght_data_h2_layer30.0_shrinking_final_pyro_cascades_simulation350.xyz
123 / 161 file name: bonds_lenght_data_h2_layer39.0_shrinking_final_pyro_cascades_simulation350.xyz
124 / 161 file name: bonds_lenght_data_h2_layer113.0_shrinking_final_pyro_cascades_simulation350